# Feature Engineering - Dataset de texto (tabular)

Este notebook implementa el plan completo de ingeniería de características para el dataset tabular de datos clínicos veterinarios.

**Objetivo:** Crear características derivadas que mejoren el poder predictivo de los modelos de ML.

**Modelos objetivo:** Regresión Logística, SVM Lineal, Random Forest

## 1. Setup y Carga de Datos

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Cargar dataset con target
df = pd.read_csv('veterinary_clinical_dataset_with_target.csv')

print(f"Dataset cargado: {df.shape}")
print(f"\nColumnas originales:")
print(df.columns.tolist())

Dataset cargado: (5013, 17)

Columnas originales:
['AnimalName', 'Breed', 'Age', 'Weight_kg', 'MedicalHistory', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5', 'MedicalHistory_norm', 'Symptom_1_norm', 'Symptom_2_norm', 'Symptom_3_norm', 'Symptom_4_norm', 'Symptom_5_norm', 'target_derm']


## 2. División Train/Valid/Test

Dividimos el dataset antes de cualquier transformación para evitar data leakage.
- Train: 70%
- Valid: 15%
- Test: 15%

Estratificación por `target_derm` y, si es posible, por raza.

In [2]:
# Seleccionar columnas base (sin las normalizadas)
base_cols = ['AnimalName', 'Breed', 'Age', 'Weight_kg', 'MedicalHistory',
             'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5', 'target_derm']
df_base = df[base_cols].copy()

# Split train/temp (70/30)
train_df, temp_df = train_test_split(
    df_base, 
    test_size=0.30, 
    random_state=42, 
    stratify=df_base['target_derm']
)

# Split temp en valid/test (50/50 del 30% = 15% cada uno)
valid_df, test_df = train_test_split(
    temp_df, 
    test_size=0.50, 
    random_state=42, 
    stratify=temp_df['target_derm']
)

print(f"Train: {train_df.shape[0]} ({train_df.shape[0]/len(df_base)*100:.1f}%)")
print(f"Valid: {valid_df.shape[0]} ({valid_df.shape[0]/len(df_base)*100:.1f}%)")
print(f"Test:  {test_df.shape[0]} ({test_df.shape[0]/len(df_base)*100:.1f}%)")

print("\nBalance de target_derm en cada split:")
print("Train:", train_df['target_derm'].value_counts(normalize=True).to_dict())
print("Valid:", valid_df['target_derm'].value_counts(normalize=True).to_dict())
print("Test: ", test_df['target_derm'].value_counts(normalize=True).to_dict())

Train: 3509 (70.0%)
Valid: 752 (15.0%)
Test:  752 (15.0%)

Balance de target_derm en cada split:
Train: {0: 0.7965232259903107, 1: 0.20347677400968936}
Valid: {0: 0.7965425531914894, 1: 0.20345744680851063}
Test:  {0: 0.7965425531914894, 1: 0.20345744680851063}


## Feature Engineering - Bloque 1: Variables Numéricas

### Edad (Age)

In [3]:
def create_age_features(df):
    """Crea características derivadas de Age."""
    df = df.copy()
    
    # Age buckets (cachorro, adulto, senior)
    df['age_bucket'] = pd.cut(
        df['Age'], 
        bins=[0, 1, 7, 100], 
        labels=['puppy', 'adult', 'senior']
    )
    
    # Age squared (para capturar relaciones no lineales)
    df['age_squared'] = df['Age'] ** 2
    
    return df

train_df = create_age_features(train_df)
valid_df = create_age_features(valid_df)
test_df = create_age_features(test_df)

print("Distribución de age_bucket en train:")
print(train_df['age_bucket'].value_counts())

Distribución de age_bucket en train:
age_bucket
senior    1885
adult     1422
puppy      202
Name: count, dtype: int64


### Peso (Weight_kg) y Peso Relativo por Raza

In [4]:
def create_weight_features(train, valid, test):
    """Crea características de peso, incluyendo z-score por raza."""
    train = train.copy()
    valid = valid.copy()
    test = test.copy()
    
    # Calcular estadísticas de peso por raza SOLO en train
    breed_weight_stats = train.groupby('Breed')['Weight_kg'].agg(['mean', 'std']).reset_index()
    breed_weight_stats.columns = ['Breed', 'breed_weight_mean', 'breed_weight_std']
    
    # Merge con cada split
    for split_df in [train, valid, test]:
        split_df_merged = split_df.merge(breed_weight_stats, on='Breed', how='left')
        
        # Z-score de peso por raza
        split_df_merged['weight_zscore_by_breed'] = (
            (split_df_merged['Weight_kg'] - split_df_merged['breed_weight_mean']) / 
            split_df_merged['breed_weight_std']
        )
        
        # Rellenar NaN (razas sin suficientes datos) con 0
        split_df_merged['weight_zscore_by_breed'].fillna(0, inplace=True)
        
        # Actualizar el dataframe original
        if split_df is train:
            train = split_df_merged
        elif split_df is valid:
            valid = split_df_merged
        else:
            test = split_df_merged
    
    # Interacción Age × Weight
    train['age_weight_interaction'] = train['Age'] * train['Weight_kg']
    valid['age_weight_interaction'] = valid['Age'] * valid['Weight_kg']
    test['age_weight_interaction'] = test['Age'] * test['Weight_kg']
    
    return train, valid, test

train_df, valid_df, test_df = create_weight_features(train_df, valid_df, test_df)

print("Estadísticas de weight_zscore_by_breed en train:")
print(train_df['weight_zscore_by_breed'].describe())

Estadísticas de weight_zscore_by_breed en train:
count    3.509000e+03
mean     5.872254e-17
std      9.987164e-01
min     -2.316014e+00
25%     -8.193545e-01
50%     -7.331995e-04
75%      7.761314e-01
max      2.603509e+00
Name: weight_zscore_by_breed, dtype: float64


## Feature Engineering - Bloque 2: Variables Categóricas

### Raza (Breed)

In [5]:
def create_breed_features(train, valid, test, top_n=10):
    """Agrupa razas raras y crea flag breed_other."""
    train = train.copy()
    valid = valid.copy()
    test = test.copy()
    
    # Identificar top N razas en train
    top_breeds = train['Breed'].value_counts().head(top_n).index.tolist()
    
    # Crear breed_grouped
    train['breed_grouped'] = train['Breed'].apply(lambda x: x if x in top_breeds else 'other')
    valid['breed_grouped'] = valid['Breed'].apply(lambda x: x if x in top_breeds else 'other')
    test['breed_grouped'] = test['Breed'].apply(lambda x: x if x in top_breeds else 'other')
    
    # Flag breed_other
    train['breed_is_other'] = (train['breed_grouped'] == 'other').astype(int)
    valid['breed_is_other'] = (valid['breed_grouped'] == 'other').astype(int)
    test['breed_is_other'] = (test['breed_grouped'] == 'other').astype(int)
    
    print(f"Top {top_n} razas: {top_breeds}")
    print(f"\nDistribución breed_grouped en train:")
    print(train['breed_grouped'].value_counts())
    
    return train, valid, test

train_df, valid_df, test_df = create_breed_features(train_df, valid_df, test_df, top_n=10)

Top 10 razas: ['mixed breed', 'rottweiler', 'bulldog', 'german shepherd', 'boxer', 'yorkshire terrier', 'beagle', 'labrador retriever', 'poodle', 'golden retriever']

Distribución breed_grouped en train:
breed_grouped
mixed breed           382
rottweiler            378
bulldog               378
german shepherd       373
boxer                 351
yorkshire terrier     339
beagle                337
labrador retriever    336
poodle                327
golden retriever      308
Name: count, dtype: int64


### Historial Médico (MedicalHistory)

In [6]:
def norm_txt(s):
    """Normaliza texto."""
    if pd.isna(s):
        return ''
    return str(s).strip().lower()

def create_medical_history_features(df):
    """Crea flags binarios para patrones en historial médico."""
    df = df.copy()
    
    df['MedicalHistory_norm'] = df['MedicalHistory'].apply(norm_txt)
    
    # Flags por condición
    conditions = {
        'is_vaccinated': 'vaccinated',
        'has_parasite_history': 'parasite',
        'has_chronic_illness': 'chronic',
        'has_allergies': 'allergies',
        'has_skin_history': 'skin',
        'has_heart_history': 'heart',
        'has_dental_issues': 'dental',
        'has_kidney_history': 'kidney',
        'has_surgery_history': 'surgery',
        'no_previous_conditions': 'no previous'
    }
    
    for flag_name, keyword in conditions.items():
        df[flag_name] = df['MedicalHistory_norm'].str.contains(keyword, na=False).astype(int)
    
    # Conteo de condiciones crónicas/repetitivas
    chronic_flags = ['has_chronic_illness', 'has_skin_history', 'has_heart_history', 
                     'has_kidney_history', 'has_allergies']
    df['chronic_conditions_count'] = df[chronic_flags].sum(axis=1)
    
    return df

train_df = create_medical_history_features(train_df)
valid_df = create_medical_history_features(valid_df)
test_df = create_medical_history_features(test_df)

print("Flags de historial médico creados:")
hist_flags = [c for c in train_df.columns if c.startswith('is_') or c.startswith('has_') or c.startswith('no_')]
print(train_df[hist_flags].sum().sort_values(ascending=False))

Flags de historial médico creados:
is_vaccinated             622
has_skin_history          364
no_previous_conditions    357
has_surgery_history       353
has_chronic_illness       340
has_parasite_history      309
has_allergies             309
has_dental_issues         292
has_kidney_history        288
has_heart_history         275
dtype: int64


## Feature Engineering - Bloque 3: Síntomas

### Normalización y Multi-Hot Encoding

In [7]:
def normalize_symptoms(df):
    """Normaliza columnas de síntomas."""
    df = df.copy()
    symptom_cols = [f'Symptom_{i}' for i in range(1, 6)]
    
    for col in symptom_cols:
        df[col + '_norm'] = df[col].apply(norm_txt)
    
    return df

train_df = normalize_symptoms(train_df)
valid_df = normalize_symptoms(valid_df)
test_df = normalize_symptoms(test_df)

# Obtener todos los síntomas únicos del train
symptom_cols_norm = [f'Symptom_{i}_norm' for i in range(1, 6)]
all_symptoms_train = set()
for col in symptom_cols_norm:
    all_symptoms_train.update(train_df[col].dropna().unique())

all_symptoms_train.discard('')  # Remover vacíos

# Contar frecuencia de cada síntoma
symptom_freq = {}
for symptom in all_symptoms_train:
    count = sum((train_df[col] == symptom).sum() for col in symptom_cols_norm)
    symptom_freq[symptom] = count

# Top 30 síntomas más frecuentes
top_symptoms = sorted(symptom_freq.items(), key=lambda x: x[1], reverse=True)[:30]
top_symptom_names = [s[0] for s in top_symptoms]

print(f"Total de síntomas únicos en train: {len(all_symptoms_train)}")
print(f"\nTop 30 síntomas más frecuentes:")
for symptom, count in top_symptoms:
    print(f"  {symptom}: {count}")

Total de síntomas únicos en train: 828

Top 30 síntomas más frecuentes:
  fever: 1289
  diarrhea: 1071
  weight loss: 1056
  pain: 983
  coughing: 951
  lethargy: 932
  weakness: 912
  vomiting: 895
  anorexia: 826
  sneezing: 819
  pains: 252
  loss of appetite: 206
  death: 195
  depression: 148
  lameness: 122
  nasal discharge: 104
  uteria inertia: 87
  fetopelvic dispropotion: 86
  malpresentation: 84
  swelling: 75
  blindness: 58
  dehydration: 54
  dyspnea: 51
  difficulty in breathing: 48
  dullness: 41
  poor appetite: 41
  skin rashes: 39
  difficulty breathing: 37
  emaciation: 36
  ruffled feathers: 36


In [8]:
def create_symptom_multihot(df, symptom_list):
    """Crea columnas multi-hot para síntomas."""
    df = df.copy()
    symptom_cols_norm = [f'Symptom_{i}_norm' for i in range(1, 6)]
    
    for symptom in symptom_list:
        col_name = f'has_{symptom.replace(" ", "_")}'
        df[col_name] = 0
        for scol in symptom_cols_norm:
            df[col_name] = df[col_name] | (df[scol] == symptom).astype(int)
    
    return df

train_df = create_symptom_multihot(train_df, top_symptom_names)
valid_df = create_symptom_multihot(valid_df, top_symptom_names)
test_df = create_symptom_multihot(test_df, top_symptom_names)

print(f"\nColumnas multi-hot de síntomas creadas: {len(top_symptom_names)}")


Columnas multi-hot de síntomas creadas: 30


### Categorías Clínicas de Síntomas

In [9]:
def create_symptom_categories(df):
    """Crea contadores por categoría clínica de síntomas."""
    df = df.copy()
    
    # Definir categorías clínicas
    categories = {
        'gi': ['diarrhea', 'vomiting', 'anorexia', 'abdominal pain', 'indigestion', 'nausea', 'constipation'],
        'respiratory': ['coughing', 'sneezing', 'noisy breathing', 'outstretched neck', 'breathing difficulty'],
        'systemic': ['fever', 'weakness', 'lethargy', 'weight loss', 'pain', 'loss of appetite'],
        'neurological': ['muscle twitching', 'shivering', 'tremor', 'seizure', 'paralysis'],
        'dermatological': ['dermatitis', 'ringworm', 'skin', 'lesion', 'rash', 'alopecia', 'pruritus', 'itch']
    }
    
    symptom_cols_norm = [f'Symptom_{i}_norm' for i in range(1, 6)]
    
    for cat_name, keywords in categories.items():
        count_col = f'{cat_name}_symptoms_count'
        df[count_col] = 0
        
        for scol in symptom_cols_norm:
            for kw in keywords:
                df[count_col] += df[scol].str.contains(kw, na=False).astype(int)
    
    return df

train_df = create_symptom_categories(train_df)
valid_df = create_symptom_categories(valid_df)
test_df = create_symptom_categories(test_df)

print("Contadores de categorías clínicas creados:")
cat_cols = [c for c in train_df.columns if c.endswith('_symptoms_count')]
print(train_df[cat_cols].describe())

Contadores de categorías clínicas creados:
       gi_symptoms_count  respiratory_symptoms_count  systemic_symptoms_count  \
count        3509.000000                 3509.000000              3509.000000   
mean            0.829866                    0.510402                 1.667427   
std             0.946010                    0.727547                 1.078832   
min             0.000000                    0.000000                 0.000000   
25%             0.000000                    0.000000                 1.000000   
50%             1.000000                    0.000000                 2.000000   
75%             1.000000                    1.000000                 2.000000   
max             5.000000                    4.000000                 5.000000   

       neurological_symptoms_count  dermatological_symptoms_count  
count                  3509.000000                    3509.000000  
mean                      0.025933                       0.126247  
std                    

### Conteo Total de Síntomas y Co-ocurrencias

In [10]:
def create_symptom_aggregates(df):
    """Crea conteo total de síntomas y co-ocurrencias clínicas."""
    df = df.copy()
    symptom_cols_norm = [f'Symptom_{i}_norm' for i in range(1, 6)]
    
    # Conteo total de síntomas (no vacíos)
    df['total_symptoms_count'] = (df[symptom_cols_norm] != '').sum(axis=1)
    
    # Co-ocurrencias clínicas relevantes
    cooccurrences = [
        ('fever', 'diarrhea', 'fever_and_diarrhea'),
        ('fever', 'vomiting', 'fever_and_vomiting'),
        ('weight loss', 'anorexia', 'weightloss_and_anorexia'),
        ('pain', 'vomiting', 'pain_and_vomiting'),
        ('coughing', 'sneezing', 'coughing_and_sneezing'),
        ('weakness', 'weight loss', 'weakness_and_weightloss')
    ]
    
    for symp1, symp2, col_name in cooccurrences:
        has_symp1 = sum((df[col].str.contains(symp1, na=False)).astype(int) for col in symptom_cols_norm) > 0
        has_symp2 = sum((df[col].str.contains(symp2, na=False)).astype(int) for col in symptom_cols_norm) > 0
        df[col_name] = (has_symp1 & has_symp2).astype(int)
    
    return df

train_df = create_symptom_aggregates(train_df)
valid_df = create_symptom_aggregates(valid_df)
test_df = create_symptom_aggregates(test_df)

print("Total symptoms count:")
print(train_df['total_symptoms_count'].value_counts().sort_index())

print("\nCo-ocurrencias de síntomas:")
cooc_cols = [c for c in train_df.columns if '_and_' in c]
print(train_df[cooc_cols].sum().sort_values(ascending=False))

Total symptoms count:
total_symptoms_count
5    3509
Name: count, dtype: int64

Co-ocurrencias de síntomas:
fever_and_diarrhea         294
pain_and_vomiting          276
weakness_and_weightloss    223
weightloss_and_anorexia    222
coughing_and_sneezing      198
fever_and_vomiting         196
dtype: int64


## One-Hot Encoding de Variables Categóricas

Aplicamos one-hot encoding a `breed_grouped` y `age_bucket`.

In [11]:
def apply_onehot_encoding(train, valid, test):
    """Aplica one-hot encoding a variables categóricas."""
    # Columnas a codificar
    cat_cols = ['breed_grouped', 'age_bucket']
    
    # One-hot en train
    train_encoded = pd.get_dummies(train, columns=cat_cols, prefix=cat_cols, drop_first=False)
    
    # Obtener las columnas creadas
    onehot_cols = [c for c in train_encoded.columns if any(c.startswith(cat + '_') for cat in cat_cols)]
    
    # Aplicar a valid y test (asegurando mismas columnas)
    valid_encoded = pd.get_dummies(valid, columns=cat_cols, prefix=cat_cols, drop_first=False)
    test_encoded = pd.get_dummies(test, columns=cat_cols, prefix=cat_cols, drop_first=False)
    
    # Alinear columnas
    for col in onehot_cols:
        if col not in valid_encoded.columns:
            valid_encoded[col] = 0
        if col not in test_encoded.columns:
            test_encoded[col] = 0
    
    return train_encoded, valid_encoded, test_encoded, onehot_cols

train_df, valid_df, test_df, onehot_cols = apply_onehot_encoding(train_df, valid_df, test_df)

print(f"Columnas one-hot creadas: {len(onehot_cols)}")
print(f"Ejemplos: {onehot_cols[:10]}")

Columnas one-hot creadas: 13
Ejemplos: ['breed_grouped_beagle', 'breed_grouped_boxer', 'breed_grouped_bulldog', 'breed_grouped_german shepherd', 'breed_grouped_golden retriever', 'breed_grouped_labrador retriever', 'breed_grouped_mixed breed', 'breed_grouped_poodle', 'breed_grouped_rottweiler', 'breed_grouped_yorkshire terrier']


## Selección de Features Finales y Limpieza

In [12]:
# Columnas a excluir del dataset final
cols_to_drop = [
    'AnimalName', 'Breed', 'MedicalHistory', 
    'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5',
    'MedicalHistory_norm',
    'Symptom_1_norm', 'Symptom_2_norm', 'Symptom_3_norm', 'Symptom_4_norm', 'Symptom_5_norm',
    'breed_weight_mean', 'breed_weight_std'  # Columnas auxiliares
]

# Remover columnas que existan
cols_to_drop_existing = [c for c in cols_to_drop if c in train_df.columns]

train_final = train_df.drop(columns=cols_to_drop_existing)
valid_final = valid_df.drop(columns=cols_to_drop_existing)
test_final = test_df.drop(columns=cols_to_drop_existing)

print(f"Dimensiones finales:")
print(f"  Train: {train_final.shape}")
print(f"  Valid: {valid_final.shape}")
print(f"  Test:  {test_final.shape}")

print(f"\nTotal de features creadas: {train_final.shape[1] - 1}")

Dimensiones finales:
  Train: (3509, 73)
  Valid: (752, 73)
  Test:  (752, 73)

Total de features creadas: 72


## Guardar Datasets Featurizados

In [13]:
# Guardar datasets
train_final.to_csv('clinical_features_train.csv', index=False)
valid_final.to_csv('clinical_features_valid.csv', index=False)
test_final.to_csv('clinical_features_test.csv', index=False)

print("✓ Datasets guardados:")
print("  - clinical_features_train.csv")
print("  - clinical_features_valid.csv")
print("  - clinical_features_test.csv")

✓ Datasets guardados:
  - clinical_features_train.csv
  - clinical_features_valid.csv
  - clinical_features_test.csv


## Diccionario de Features

Generamos un diccionario documentando todas las features creadas.

In [14]:
# Crear diccionario de features
feature_dict = []

# Features numéricas originales
feature_dict.append({'feature': 'Age', 'type': 'numeric', 'source': 'original', 'description': 'Edad del animal en años'})
feature_dict.append({'feature': 'Weight_kg', 'type': 'numeric', 'source': 'original', 'description': 'Peso del animal en kg'})

# Features derivadas de edad
feature_dict.append({'feature': 'age_squared', 'type': 'numeric', 'source': 'derived', 'description': 'Age^2 para capturar relaciones no lineales'})

# Features derivadas de peso
feature_dict.append({'feature': 'weight_zscore_by_breed', 'type': 'numeric', 'source': 'derived', 'description': 'Z-score de peso dentro de cada raza'})
feature_dict.append({'feature': 'age_weight_interaction', 'type': 'numeric', 'source': 'derived', 'description': 'Interacción Age × Weight_kg'})

# Features de raza
feature_dict.append({'feature': 'breed_is_other', 'type': 'binary', 'source': 'derived', 'description': '1 si la raza no está en el top 10'})

# Features de historial médico
hist_features = ['is_vaccinated', 'has_parasite_history', 'has_chronic_illness', 'has_allergies',
                'has_skin_history', 'has_heart_history', 'has_dental_issues', 'has_kidney_history',
                'has_surgery_history', 'no_previous_conditions', 'chronic_conditions_count']
for feat in hist_features:
    ftype = 'binary' if feat != 'chronic_conditions_count' else 'numeric'
    feature_dict.append({'feature': feat, 'type': ftype, 'source': 'derived', 'description': f'Flag/count de historial médico'})

# Features de síntomas
feature_dict.append({'feature': 'total_symptoms_count', 'type': 'numeric', 'source': 'derived', 'description': 'Número total de síntomas reportados'})

cat_features = ['gi_symptoms_count', 'respiratory_symptoms_count', 'systemic_symptoms_count',
               'neurological_symptoms_count', 'dermatological_symptoms_count']
for feat in cat_features:
    feature_dict.append({'feature': feat, 'type': 'numeric', 'source': 'derived', 'description': f'Conteo de síntomas en categoría {feat.split("_")[0]}'})

# Target
feature_dict.append({'feature': 'target_derm', 'type': 'binary', 'source': 'target', 'description': 'Variable objetivo: 1=dermatológico, 0=no dermatológico'})

# Convertir a DataFrame
feature_dict_df = pd.DataFrame(feature_dict)

# Guardar
feature_dict_df.to_excel('feature_dictionary.xlsx', index=False)

print("✓ Diccionario de features guardado: feature_dictionary.xlsx")
print(f"\nTotal de features documentadas: {len(feature_dict_df)}")
feature_dict_df.head(20)

✓ Diccionario de features guardado: feature_dictionary.xlsx

Total de features documentadas: 24


,feature,type,source,description
0,Age,numeric,original,Edad del animal en años
1,Weight_kg,numeric,original,Peso del animal en kg
2,age_squared,numeric,derived,Age^2 para capturar relaciones no lineales
3,weight_zscore_by_breed,numeric,derived,Z-score de peso dentro de cada raza
4,age_weight_interaction,numeric,derived,Interacción Age × Weight_kg
5,breed_is_other,binary,derived,1 si la raza no está en el top 10
6,is_vaccinated,binary,derived,Flag/count de historial médico
7,has_parasite_history,binary,derived,Flag/count de historial médico
8,has_chronic_illness,binary,derived,Flag/count de historial médico
9,has_allergies,binary,derived,Flag/count de historial médico


## 10. Resumen y Próximos Pasos

1. Splits creados:

   - Train: 3509 muestras

   - Valid: 752 muestras

   - Test:  752 muestras

2. Features totales: 72 (+ target)

3. Bloques de features creados:

   - Numéricas derivadas: age_squared, weight_zscore_by_breed, age_weight_interaction

   - Raza: breed_grouped (one-hot), breed_is_other

   - Historial médico: 10 flags + chronic_conditions_count

   - Síntomas multi-hot: top 30 síntomas

   - Categorías clínicas: 5 contadores (GI, respiratorio, sistémico, neuro, derm)

   - Agregados: total_symptoms_count + 6 co-ocurrencias

   - One-hot: age_bucket (puppy/adult/senior)

4. Archivos generados:

   clinical_features_train.csv

   clinical_features_valid.csv

   clinical_features_test.csv

   feature_dictionary.xlsx

5. Próximos pasos:

   → Entrenar modelos baseline (Logistic Regression, SVM, Random Forest)

   → Evaluar con F1 macro, balanced accuracy, AUC

   → Aplicar SHAP/permutation importance para interpretabilidad

   → Ajustar hiperparámetros con GridSearch/RandomSearch

   → Considerar class_weight='balanced' por desbalance 80/20